# 08_IMPORT_TO_DB

Targeted competencies:
- **C3** — Final aggregation/cleaning pass before storage: deduplication and format standardization across the merged API + IMDb data-file sources.
- **C4** — Functional import into a relational database created from the physical model (`schema.sql`).

**Prerequisites**
- `schema.sql` already executed on the target MySQL server (creates the `streaming_marketing` database and its tables).
- `DATA/PROCESSED/all_streaming_titles_enriched.csv` already produced by `02B_IMDB_DATASET_ENRICHMENT.ipynb`.

**Credentials**: never hardcode them. Set environment variables before launching Jupyter, or set them in the cell below for this session only (do not commit real values if you save this notebook).

In [ ]:
%pip install -q pandas mysql-connector-python python-dotenv

In [ ]:
import os
import logging
import pandas as pd
import mysql.connector
from mysql.connector import Error as MySQLError

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
log = logging.getLogger(__name__)

ENRICHED_PATH = "DATA/PROCESSED/all_streaming_titles_enriched.csv"

## 1. Database connection settings

Set these for the current session. Prefer environment variables (`os.environ.get(...)`) over hardcoding a real password directly here — especially if this notebook will be pushed to GitHub.

In [ ]:
from dotenv import load_dotenv

load_dotenv()  # reads the .env file in the project root

DB_HOST = os.environ.get("DB_HOST", "localhost")
DB_USER = os.environ.get("DB_USER", "root")
DB_PASSWORD = os.environ.get("DB_PASSWORD", "")
DB_NAME = os.environ.get("DB_NAME", "streaming_marketing")

def get_connection():
    return mysql.connector.connect(
        host=DB_HOST,
        user=DB_USER,
        password=DB_PASSWORD,
        database=DB_NAME,
    )

## 2. Load and do a final cleaning pass (C3)

- Standardizes column names (lowercase, no spaces/dashes) so they are safe SQL identifiers.
- Removes duplicate titles and rows missing a title or content type.
- Prefers the `imdb_id` resolved during the IMDb data-file enrichment step over the original one.

In [ ]:
def load_and_clean() -> pd.DataFrame:
    log.info(f"Loading enriched dataset: {ENRICHED_PATH}")
    df = pd.read_csv(ENRICHED_PATH, low_memory=False, encoding="utf-8-sig")

    # Column-name standardization (carried over from 07_SQL.ipynb):
    # lowercase, no spaces/dashes, so column names are safe to use as
    # SQL identifiers and consistent across the whole pipeline.
    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(" ", "_")
        .str.replace("-", "_")
    )

    before = len(df)
    df = df.drop_duplicates(subset=["title", "release_year", "content_type"], keep="first")
    df = df.dropna(subset=["title", "content_type"])
    df["content_type"] = df["content_type"].str.lower().str.strip()

    # Prefer the resolved imdb_id from the data-file matching step,
    # fall back to the original one if it was already valid.
    if "imdb_resolved_id" in df.columns:
        df["imdb_id"] = df["imdb_resolved_id"].where(
            df["imdb_resolved_id"].notna() & (df["imdb_resolved_id"] != ""),
            df.get("imdb_id"),
        )

    log.info(f"Cleaning pass: {before} -> {len(df)} rows kept.")
    return df

dataset = load_and_clean()
dataset.head(3)

## 3. Helper functions for the import (C4)

- `get_or_create` — generic lookup-or-insert for dimension tables (`LANGUAGE_TABLE`, `GENRE`).
- `get_or_create_person` / `insert_people` — same idea for `PERSON`/`CONTENT_PERSON`, splitting the `imdb_cast` / `imdb_directors` / `imdb_cinematographers` pipe-separated strings produced by the enrichment notebook into normalized rows with a `role`.

In [ ]:
def get_or_create(cursor, table, id_col, name_col, value):
    cursor.execute(f"SELECT {id_col} FROM {table} WHERE {name_col} = %s", (value,))
    row = cursor.fetchone()
    if row:
        return row[0]
    cursor.execute(f"INSERT INTO {table} ({name_col}) VALUES (%s)", (value,))
    return cursor.lastrowid


def get_or_create_person(cursor, full_name: str) -> int:
    cursor.execute("SELECT id_person FROM PERSON WHERE full_name = %s", (full_name,))
    row = cursor.fetchone()
    if row:
        return row[0]
    cursor.execute("INSERT INTO PERSON (full_name) VALUES (%s)", (full_name,))
    return cursor.lastrowid


def insert_people(cursor, id_content: int, names_field: str, role: str):
    """names_field is a ' | '-joined string, as produced by the enrichment notebook."""
    if not names_field or pd.isna(names_field):
        return
    names = [n.strip() for n in str(names_field).split("|") if n.strip()]
    for order, name in enumerate(names, start=1):
        id_person = get_or_create_person(cursor, name)
        cursor.execute(
            """INSERT IGNORE INTO CONTENT_PERSON (id_content, id_person, role, ordering)
               VALUES (%s,%s,%s,%s)""",
            (id_content, id_person, role, order),
        )

## 4. Functional import into MySQL (C4)

Inserts every row into `CONTENT`, links its genres via `CONTENT_GENRE`, writes its four derived metrics into `SCORE`, and writes its cast/directors/cinematographers into `PERSON` + `CONTENT_PERSON`. Rows that fail (e.g. a constraint violation) are logged and skipped rather than aborting the whole run.

In [ ]:
def import_dataframe(df: pd.DataFrame):
    conn = None
    try:
        conn = get_connection()
        cursor = conn.cursor()
        log.info("MySQL connection established.")

        inserted, skipped = 0, 0
        for _, row in df.iterrows():
            try:
                id_language = None
                if pd.notna(row.get("original_language")):
                    id_language = get_or_create(
                        cursor, "LANGUAGE_TABLE", "id_language", "language_code", row["original_language"]
                    )

                cursor.execute(
                    """INSERT INTO CONTENT
                       (title, original_title, content_type, id_language, release_year,
                        popularity, vote_average, vote_count, overview, imdb_id,
                        imdb_match_status, adult, source_origin)
                       VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s)""",
                    (
                        row["title"], row.get("original_title"), row["content_type"], id_language,
                        row.get("release_year"), row.get("popularity"), row.get("vote_average"),
                        row.get("vote_count"), row.get("overview"), row.get("imdb_id"),
                        row.get("imdb_match_status"), bool(row.get("adult", False)),
                        "TMDb/TVMaze API",
                    ),
                )
                id_content = cursor.lastrowid

                for genre in str(row.get("genre_names", "")).split(","):
                    genre = genre.strip()
                    if not genre:
                        continue
                    id_genre = get_or_create(cursor, "GENRE", "id_genre", "genre_name", genre)
                    cursor.execute(
                        "INSERT IGNORE INTO CONTENT_GENRE (id_content, id_genre) VALUES (%s,%s)",
                        (id_content, id_genre),
                    )

                cursor.execute(
                    """INSERT INTO SCORE
                       (id_content, visibility_score, engagement_score, reception_score, business_value_score)
                       VALUES (%s,%s,%s,%s,%s)""",
                    (
                        id_content, row.get("visibility_score"), row.get("engagement_score"),
                        row.get("audience_reception_score"), row.get("business_value_score"),
                    ),
                )

                # People credits coming from the IMDb data-file enrichment (second source)
                insert_people(cursor, id_content, row.get("imdb_cast"), "cast")
                insert_people(cursor, id_content, row.get("imdb_directors"), "director")
                insert_people(cursor, id_content, row.get("imdb_cinematographers"), "cinematographer")

                inserted += 1
            except MySQLError as e:
                skipped += 1
                log.warning(f"Row skipped ({row.get('title')}): {e}")

        conn.commit()
        log.info(f"Import complete: {inserted} rows inserted, {skipped} skipped.")

    except MySQLError as e:
        log.error(f"MySQL connection/import error: {e}")
        if conn:
            conn.rollback()
    finally:
        if conn and conn.is_connected():
            conn.close()
            log.info("MySQL connection closed.")

## 5. Run the import

In [ ]:
import_dataframe(dataset)

## 6. Quick verification

Sanity check: confirm rows actually landed in each table.

In [ ]:
conn = get_connection()
cursor = conn.cursor()
for table in ["CONTENT", "GENRE", "CONTENT_GENRE", "SCORE", "PERSON", "CONTENT_PERSON", "LANGUAGE_TABLE"]:
    cursor.execute(f"SELECT COUNT(*) FROM {table}")
    print(table, "->", cursor.fetchone()[0], "rows")
conn.close()